# 🎯 Arabic CV Analyzer — v4
## الإصلاحات المستهدفة بناءً على نتائج v3

### ما اكتشفناه في v3:
```
ضعيف  < 68.0
متوسط < 82.0
جيد   < 82.0  ← ❌ T2 = T3 = 82 (class فاضية!)
ممتاز ≥ 82.0
```
الـ scores متمركزة في نطاق ضيق (60-90) → model مش شايف فروق

### الإصلاحات في v4:
| Fix | التفاصيل |
|-----|----------|
| ✅ Smart Thresholds | Minimum gap enforcement + تشخيص التوزيع |
| ✅ Score Rescaling | تحويل الـ range الفعلي لـ [0,1] بدل [0,100] |
| ✅ Dual Loss | Regression + Classification مع class weights |
| ✅ كل تحسينات v3 | Freeze/Unfreeze + Layer-wise LR + Ensemble |

---
## 1️⃣ Imports & Config

In [ ]:
import re, warnings, json, pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    mean_absolute_error, mean_squared_error, r2_score
)
from scipy.stats import spearmanr

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.facecolor':'white','axes.facecolor':'#f8f9fa',
    'axes.grid':True,'grid.alpha':0.4,
    'axes.spines.top':False,'axes.spines.right':False,
    'font.size':11,'axes.titlesize':13,'axes.titleweight':'bold',
})
CLRS = {'AraBERT':'#2563EB','CAMeL-BERT':'#EA580C','Ensemble':'#7C3AED'}

device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_DIR = Path('saved_model')
SAVE_DIR.mkdir(exist_ok=True)
print(f'🖥️  Device: {device} | PyTorch: {torch.__version__}')

MODELS = {
    'AraBERT'   : 'aubmindlab/bert-base-arabertv2',
    'CAMeL-BERT': 'CAMeL-Lab/bert-base-arabic-camelbert-mix'
}
CLASS_NAMES  = ['ضعيف','متوسط','جيد','ممتاز']
CLASS_LABELS = [0,1,2,3]

BEST_PARAMS = {
    'lr'            : 1e-5,
    'head_lr'       : 1e-4,
    'batch_size'    : 8,
    'dropout'       : 0.2,
    'max_grad_norm' : 1.0,
    'warmup_ratio'  : 0.1,
    'weight_decay'  : 0.01,
    'lr_decay'      : 0.9,
    'freeze_epochs' : 2,
    'aux_weight'    : 0.3,   # وزن الـ classification loss جوّا الـ dual loss
}
FULL_EPOCHS = 10
PATIENCE    = 3
print('✅ Config loaded')

---
## 2️⃣ Data Loading

In [ ]:
df = pd.read_csv('arabic_cvs_with_scores.csv', encoding='utf-8-sig')
df['cv_words'] = df['arabic_cv'].astype(str).apply(lambda x: len(x.split()))
print(f'✅ {len(df):,} CVs')
display(df.head(3))

---
## 3️⃣ ATS Engine + Preprocessing

In [ ]:
DOMAIN_KEYWORDS = {
    'تكنولوجيا المعلومات': ['python','java','javascript','sql','nosql','react','node','docker','kubernetes','aws','azure','git','api','rest','machine learning','deep learning','linux','agile','scrum','ci/cd'],
    'موارد بشرية'        : ['توظيف','تدريب','تطوير','أداء','رواتب','تعويضات','علاقات عمل','kpi','okr','onboarding','hris','payroll','retention','culture','succession','talent','engagement'],
    'مالية ومحاسبة'      : ['excel','sap','oracle','ifrs','gaap','ميزانية','تدقيق','ضرائب','تحليل مالي','تقارير','quickbooks','erp','cpa','cma','dcf'],
    'هندسة'              : ['autocad','solidworks','matlab','ansys','pmp','iso','six sigma','lean','revit','primavera','bim','حديد','خرسانة','ميكانيكا','كهرباء','plc','scada'],
    'تسويق'              : ['seo','sem','google ads','facebook ads','content','analytics','crm','hubspot','mailchimp','brand','roi','kpi','digital','social media','copywriting','strategy'],
    'مبيعات'             : ['مبيعات','عملاء','crm','salesforce','target','pipeline','negotiation','b2b','b2c','revenue','quota','cold calling','account management','upsell'],
    'رعاية صحية'         : ['سريري','مرضى','تشخيص','علاج','أدوية','مختبر','تمريض','طوارئ','جراحة','emr','his','bls','acls','hipaa'],
    'تعليم'              : ['مناهج','تدريس','طلاب','تقييم','تعليم إلكتروني','lms','moodle','خطة درس','أهداف تعليمية','bloom','differentiation'],
    'إعلام وتصميم'       : ['photoshop','illustrator','figma','adobe','premiere','after effects','typography','branding','ui','ux','indesign','3d','animation'],
    'سياحة ومطاعم'       : ['hospitality','opera','خدمة عملاء','haccp','food safety','فنادق','مطاعم','حجوزات','pos','revenue management'],
}
DEFAULT_KEYWORDS = ['خبرة','مهارات','تواصل','قيادة','تحليل','مشاريع','فريق','إدارة','تطوير','تحسين','excel','word','powerpoint']
ATS_MESSAGES = {
    'has_experience':'أضف قسم خبرة العمل بالتفصيل',
    'has_education' :'أضف قسم التعليم والمؤهلات',
    'has_skills'    :'أضف قسم المهارات التقنية والشخصية',
    'has_summary'   :'أضف ملخصاً مهنياً في بداية الـ CV',
    'has_email'     :'أضف بريدك الإلكتروني',
    'has_phone'     :'أضف رقم هاتفك',
    'good_length'   :'اجعل الـ CV بين 200-800 كلمة',
    'has_contact'   :'أضف رابط LinkedIn أو GitHub أو Portfolio'
}

def calculate_ats_score(arabic_cv, category=''):
    cv = str(arabic_cv).lower()
    checks = {
        'has_experience': any(k in cv for k in ['الخبرات','خبرة العمل','الخبرة المهنية']),
        'has_education' : any(k in cv for k in ['التعليم','المؤهلات','الدراسة']),
        'has_skills'    : any(k in cv for k in ['المهارات','الكفاءات','قدرات']),
        'has_summary'   : any(k in cv for k in ['ملخص','نبذة','عن نفسي','profile']),
        'has_email'     : bool(re.search(r'[\w\.-]+@[\w\.-]+', cv)),
        'has_phone'     : bool(re.search(r'[\+\d][\d\s\-]{8,}', cv)),
        'good_length'   : 200 <= len(cv.split()) <= 800,
        'has_contact'   : any(k in cv for k in ['linkedin','github','portfolio','موقع'])
    }
    w = {'has_experience':15,'has_education':12,'has_skills':12,'has_summary':8,
         'has_email':6,'has_phone':4,'good_length':6,'has_contact':7}
    struct_score  = sum(w[k] for k,v in checks.items() if v)
    kws           = DOMAIN_KEYWORDS.get(category, DEFAULT_KEYWORDS)
    matched       = [k for k in kws if k.lower() in cv]
    keyword_score = (len(matched)/max(len(kws),1))*30
    return min(int(struct_score+keyword_score),100), checks, matched

def clean_arabic_text(text):
    text = str(text)
    text = re.sub(r'[\u064B-\u065F]','',text)
    text = re.sub(r'[إأآا]','ا',text)
    text = re.sub(r'ة','ه',text)
    text = re.sub(r'ى','ي',text)
    text = re.sub(r'[^\w\s\u0600-\u06FF]',' ',text)
    return re.sub(r'\s+',' ',text).strip()

ats_r              = df.apply(lambda r: calculate_ats_score(r['arabic_cv'],r['category']),axis=1)
df['ats_score']    = ats_r.apply(lambda x: x[0])
df['ats_checks']   = ats_r.apply(lambda x: x[1])
df['ats_matched']  = ats_r.apply(lambda x: x[2])
df['ats_kw_count'] = df['ats_matched'].apply(len)
df['arabic_cv_clean'] = df['arabic_cv'].apply(clean_arabic_text)

def build_input(row):
    ats    = row['ats_score']
    bucket = 'ats_low' if ats<50 else ('ats_mid' if ats<75 else 'ats_high')
    return f"{row['category']} {bucket} [SEP] {row['arabic_cv_clean']}"

df['input_text'] = df.apply(build_input, axis=1)
df_clean = df.dropna(subset=['arabic_cv_clean','suitability_score']).copy()

train_df, temp_df = train_test_split(df_clean, test_size=.2, random_state=42)
val_df,   test_df = train_test_split(temp_df,  test_size=.5, random_state=42)
print(f'✅ Train:{len(train_df):,}  Val:{len(val_df):,}  Test:{len(test_df):,}')

---
## 4️⃣ 🔧 Score Diagnostics + Smart Thresholds

In [ ]:
# ── تشخيص توزيع الـ scores أولاً ──────────────────────
scores = train_df['suitability_score']
print('📊 Score Distribution Diagnostics:')
print(f'   Min={scores.min():.1f}  Max={scores.max():.1f}  Range={scores.max()-scores.min():.1f}')
print(f'   Mean={scores.mean():.1f}  Std={scores.std():.1f}  Median={scores.median():.1f}')
print(f'   P10={np.percentile(scores,10):.1f}  P25={np.percentile(scores,25):.1f}  '
      f'P50={np.percentile(scores,50):.1f}  P75={np.percentile(scores,75):.1f}  '
      f'P90={np.percentile(scores,90):.1f}')

In [ ]:
# ── ✅ Score Rescaling ─────────────────────────────────
# الداتا مش بتبدأ من 0 — بتبدأ من ~60 وتوصل ~95
# هنـ rescale الـ scores للـ [0,1] بناءً على الـ range الفعلي
# ده بيكبّر الفروق بين الـ CVs ويساعد الموديل يتعلم أحسن

SCORE_MIN = float(scores.quantile(0.01))   # نتجاهل الـ 1% outliers
SCORE_MAX = float(scores.quantile(0.99))

def scale_score(s):
    """تحويل الـ score للـ [0,1] بناءً على الـ range الفعلي"""
    return np.clip((s - SCORE_MIN) / (SCORE_MAX - SCORE_MIN), 0, 1)

def unscale_score(s):
    """رجوع من [0,1] للـ score الأصلي"""
    return s * (SCORE_MAX - SCORE_MIN) + SCORE_MIN

print(f'📐 Score Rescaling: [{SCORE_MIN:.1f}, {SCORE_MAX:.1f}] → [0, 1]')
print(f'   مثال: score=68 → {scale_score(68):.3f}')
print(f'   مثال: score=82 → {scale_score(82):.3f}')
print(f'   مثال: score=90 → {scale_score(90):.3f}')

In [ ]:
# ── ✅ Smart Thresholds ────────────────────────────────
MIN_GAP = 5.0   # أقل فرق مقبول بين threshold وتاني

raw_T1 = float(np.percentile(scores, 25))
raw_T2 = float(np.percentile(scores, 50))
raw_T3 = float(np.percentile(scores, 75))

# اكتشاف التوزيع الضيق
score_range = scores.max() - scores.min()
is_tight    = (score_range < 30) or (abs(raw_T2 - raw_T3) < MIN_GAP) or (abs(raw_T1 - raw_T2) < MIN_GAP)

if is_tight:
    print(f'⚠️  Score range ضيق ({score_range:.1f} pts) أو T2≈T3 — هنستخدم quartile spacing مضمون')
    # نقسم الـ range الفعلي لـ 4 أجزاء متساوية
    q_step = score_range / 4
    T1 = scores.min() + q_step
    T2 = scores.min() + 2*q_step
    T3 = scores.min() + 3*q_step
else:
    T1, T2, T3 = raw_T1, raw_T2, raw_T3
    print(f'✅ Percentile thresholds سليمة')

# تأكيد إن الـ gaps محترمة
T2 = max(T2, T1 + MIN_GAP)
T3 = max(T3, T2 + MIN_GAP)

print(f'\n📌 Final Thresholds:')
print(f'   ضعيف  < {T1:.1f}')
print(f'   متوسط < {T2:.1f}  (gap={T2-T1:.1f})')
print(f'   جيد   < {T3:.1f}  (gap={T3-T2:.1f})')
print(f'   ممتاز ≥ {T3:.1f}')

def score_to_class(score):
    if   score < T1: return 0
    elif score < T2: return 1
    elif score < T3: return 2
    else:            return 3

# توزيع الـ classes
df_clean['label'] = df_clean['suitability_score'].apply(score_to_class)
lc = df_clean['label'].map(dict(enumerate(CLASS_NAMES))).value_counts()

print('\n📊 Class Distribution:')
for cn in CLASS_NAMES:
    n   = lc.get(cn, 0)
    pct = n/len(df_clean)*100
    bar = '█' * int(pct/2)
    print(f'   {cn:>6}: {n:>4} ({pct:4.1f}%) {bar}')

In [ ]:
# ── Visualize ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('🔧 Score Diagnostics & Thresholds', fontweight='bold')

# 1. Distribution قبل rescaling
axes[0].hist(df_clean['suitability_score'], bins=30, color='#BFDBFE', edgecolor='white')
for t, lbl, clr in zip([T1,T2,T3],[f'T1={T1:.0f}',f'T2={T2:.0f}',f'T3={T3:.0f}'],
                        ['#EF4444','#F59E0B','#22C55E']):
    axes[0].axvline(t, color=clr, linestyle='--', lw=2, label=lbl)
axes[0].set_title('Score Distribution + Thresholds')
axes[0].set_xlabel('Score'); axes[0].legend(fontsize=9)

# 2. Class distribution
colors = ['#EF4444','#F59E0B','#22C55E','#2563EB']
vals   = [lc.get(cn,0) for cn in CLASS_NAMES]
bars   = axes[1].bar(CLASS_NAMES, vals, color=colors)
axes[1].set_title('Class Distribution')
for bar in bars:
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                 f'{int(bar.get_height())}\n({int(bar.get_height())/len(df_clean)*100:.0f}%)',
                 ha='center', fontsize=9, fontweight='bold')

# 3. Distribution بعد rescaling
rescaled = df_clean['suitability_score'].apply(scale_score)
axes[2].hist(rescaled, bins=30, color='#D1FAE5', edgecolor='white')
for t in [scale_score(T1),scale_score(T2),scale_score(T3)]:
    axes[2].axvline(t, color='gray', linestyle='--', lw=1.5, alpha=.7)
axes[2].set_title('Rescaled Score Distribution [0,1]')
axes[2].set_xlabel('Rescaled Score')

plt.tight_layout(); plt.savefig('01_diagnostics.png', dpi=150, bbox_inches='tight'); plt.show()

---
## 5️⃣ Model Architecture + ✅ Dual Loss

In [ ]:
class CVDatasetSliding(Dataset):
    def __init__(self, df, tokenizer, max_len=512, stride=384):
        self.df=df.reset_index(drop=True); self.tokenizer=tokenizer
        self.max_len=max_len; self.stride=stride
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        toks = self.tokenizer(row['input_text'],truncation=True,max_length=512,add_special_tokens=True)
        ids  = toks['input_ids']
        chunks_ids, chunks_mask = [], []
        start = 0
        while start < len(ids):
            end=min(start+self.max_len,len(ids))
            chunk=torch.tensor(ids[start:end],dtype=torch.long)
            mask=torch.ones(len(chunk),dtype=torch.long)
            pad=self.max_len-len(chunk)
            if pad>0:
                chunk=torch.cat([chunk,torch.zeros(pad,dtype=torch.long)])
                mask=torch.cat([mask,torch.zeros(pad,dtype=torch.long)])
            chunks_ids.append(chunk); chunks_mask.append(mask)
            if end==len(ids): break
            start+=self.stride
        return {
            'chunks_ids' : torch.stack(chunks_ids),
            'chunks_mask': torch.stack(chunks_mask),
            # ✅ نحفظ الـ rescaled score للـ regression
            'score'      : torch.tensor(scale_score(row['suitability_score']), dtype=torch.float),
            # ✅ ونحفظ الـ class label للـ classification
            'label'      : torch.tensor(score_to_class(row['suitability_score']), dtype=torch.long),
            'n_chunks'   : len(chunks_ids)
        }

def collate_fn(batch):
    max_c=max(b['n_chunks'] for b in batch)
    ids_l,mask_l,scores,labels=[],[],[],[]
    for b in batch:
        p=max_c-b['n_chunks']
        if p>0:
            ids_l.append(torch.cat([b['chunks_ids'],torch.zeros(p,512,dtype=torch.long)]))
            mask_l.append(torch.cat([b['chunks_mask'],torch.zeros(p,512,dtype=torch.long)]))
        else:
            ids_l.append(b['chunks_ids']); mask_l.append(b['chunks_mask'])
        scores.append(b['score']); labels.append(b['label'])
    return {
        'chunks_ids' : torch.stack(ids_l),
        'chunks_mask': torch.stack(mask_l),
        'n_chunks'   : [b['n_chunks'] for b in batch],
        'score'      : torch.stack(scores),
        'label'      : torch.stack(labels),
    }

class AttentionPooling(nn.Module):
    def __init__(self, hidden_size=768):
        super().__init__()
        self.attention=nn.Sequential(nn.Linear(hidden_size,256),nn.Tanh(),nn.Linear(256,1))
    def forward(self, cls_embeddings, n_chunks):
        pooled=[]
        for i in range(cls_embeddings.shape[0]):
            n=n_chunks[i]; emb=cls_embeddings[i,:n,:]
            att=F.softmax(self.attention(emb),dim=0)
            pooled.append((att*emb).sum(dim=0))
        return torch.stack(pooled)

class CVRegressor(nn.Module):
    def __init__(self, model_path, dropout=0.2, n_classes=4):
        super().__init__()
        self.bert     = AutoModel.from_pretrained(model_path)
        self.att_pool = AttentionPooling(768)
        self.shared   = nn.Sequential(
            nn.Linear(768,256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256,64),  nn.GELU(), nn.Dropout(dropout)
        )
        self.reg_head = nn.Sequential(nn.Linear(64,1), nn.Sigmoid())
        # ✅ Classification head إضافي — بيساعد الموديل يتعلم الحدود
        self.cls_head = nn.Linear(64, n_classes)

    def forward(self, chunks_ids, chunks_mask, n_chunks):
        B,C,L=chunks_ids.shape
        out=self.bert(input_ids=chunks_ids.view(B*C,L),
                      attention_mask=chunks_mask.view(B*C,L), output_attentions=False)
        cls    = out.last_hidden_state[:,0,:].view(B,C,768)
        pooled = self.att_pool(cls, n_chunks)
        shared = self.shared(pooled)
        score  = self.reg_head(shared).squeeze(-1)   # (B,)
        logits = self.cls_head(shared)               # (B, 4)
        return score, logits

print('✅ CVRegressor with dual head defined')

In [ ]:
# ── ✅ Dual Loss + Class Weights ───────────────────────
# نحسب class weights من الـ train data
train_labels = train_df['suitability_score'].apply(score_to_class).values
cw = compute_class_weight('balanced', classes=np.array([0,1,2,3]), y=train_labels)
class_weights_tensor = torch.tensor(cw, dtype=torch.float).to(device)

print('⚖️  Class weights (balanced):')
for cn, w in zip(CLASS_NAMES, cw):
    print(f'   {cn:>6}: {w:.3f}')

class DualLoss(nn.Module):
    """
    ✅ Regression Loss + Classification Loss مع بعض:
    total = (1 - aux_w) * HuberMSE(score) + aux_w * WeightedCE(logits, labels)
    """
    def __init__(self, class_weights, aux_weight=0.3, huber_delta=0.1):
        super().__init__()
        self.mse     = nn.MSELoss()
        self.huber   = nn.HuberLoss(delta=huber_delta)
        self.ce      = nn.CrossEntropyLoss(weight=class_weights)
        self.aux_w   = aux_weight

    def forward(self, pred_score, pred_logits, true_score, true_label):
        reg_loss = 0.7*self.mse(pred_score, true_score) + 0.3*self.huber(pred_score, true_score)
        cls_loss = self.ce(pred_logits, true_label)
        return (1 - self.aux_w)*reg_loss + self.aux_w*cls_loss

def run_epoch(model, loader, optimizer, criterion, scheduler=None, train=True, max_grad_norm=1.0):
    model.train() if train else model.eval()
    total_loss,preds_list,scores_list=[],[],[]
    t_loss_sum = 0
    ctx=torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            ids      = batch['chunks_ids'].to(device)
            mask     = batch['chunks_mask'].to(device)
            scores   = batch['score'].to(device)
            labels   = batch['label'].to(device)
            n_chunks = batch['n_chunks']

            if train: optimizer.zero_grad()
            pred_score, pred_logits = model(ids, mask, n_chunks)
            loss = criterion(pred_score, pred_logits, scores, labels)
            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                optimizer.step()
                if scheduler: scheduler.step()
            t_loss_sum += loss.item()
            # ✅ نرجع الـ predictions للـ scale الأصلي
            preds_list.extend([unscale_score(p) for p in pred_score.detach().cpu().tolist()])
            scores_list.extend([unscale_score(s) for s in scores.cpu().tolist()])

    mae      = mean_absolute_error(scores_list, preds_list)
    rmse     = np.sqrt(mean_squared_error(scores_list, preds_list))
    pred_cls = [score_to_class(p) for p in preds_list]
    true_cls = [score_to_class(s) for s in scores_list]
    acc      = sum(p==t for p,t in zip(pred_cls,true_cls))/len(true_cls)
    return {'loss':t_loss_sum/len(loader),'mae':mae,'rmse':rmse,'acc':acc,
            'pred_scores':preds_list,'true_scores':scores_list,
            'pred_classes':pred_cls,'true_classes':true_cls}

print('✅ DualLoss + run_epoch ready')

---
## 6️⃣ Training

In [ ]:
def build_optimizer(model, base_lr, head_lr, weight_decay, lr_decay):
    no_decay   = ['bias','LayerNorm.weight']
    num_layers = len(model.bert.encoder.layer)
    opt_params = []
    emb_lr = base_lr * (lr_decay ** num_layers)
    opt_params += [
        {'params':[p for n,p in model.bert.embeddings.named_parameters() if not any(nd in n for nd in no_decay)],
         'lr':emb_lr,'weight_decay':weight_decay},
        {'params':[p for n,p in model.bert.embeddings.named_parameters() if     any(nd in n for nd in no_decay)],
         'lr':emb_lr,'weight_decay':0.0},
    ]
    for i, layer in enumerate(model.bert.encoder.layer):
        layer_lr = base_lr * (lr_decay ** (num_layers - i))
        opt_params += [
            {'params':[p for n,p in layer.named_parameters() if not any(nd in n for nd in no_decay)],
             'lr':layer_lr,'weight_decay':weight_decay},
            {'params':[p for n,p in layer.named_parameters() if     any(nd in n for nd in no_decay)],
             'lr':layer_lr,'weight_decay':0.0},
        ]
    opt_params += [
        {'params':[p for n,p in model.shared.named_parameters()   if not any(nd in n for nd in no_decay)],'lr':head_lr,'weight_decay':weight_decay},
        {'params':[p for n,p in model.reg_head.named_parameters() if not any(nd in n for nd in no_decay)],'lr':head_lr,'weight_decay':weight_decay},
        {'params':[p for n,p in model.cls_head.named_parameters() if not any(nd in n for nd in no_decay)],'lr':head_lr,'weight_decay':weight_decay},
        {'params':model.att_pool.parameters(),'lr':head_lr,'weight_decay':weight_decay},
    ]
    return torch.optim.AdamW(opt_params)


final_models  = {}
final_history = {}
tokenizers    = {}

for model_name, model_path in MODELS.items():
    print(f'\n{"="*60}\n🚀 Training: {model_name}\n{"="*60}')

    tok = AutoTokenizer.from_pretrained(model_path)
    tokenizers[model_name] = tok

    train_ds = CVDatasetSliding(train_df, tok)
    val_ds   = CVDatasetSliding(val_df,   tok)
    train_dl = DataLoader(train_ds, batch_size=BEST_PARAMS['batch_size'], shuffle=True, collate_fn=collate_fn)
    val_dl   = DataLoader(val_ds,   batch_size=BEST_PARAMS['batch_size'], collate_fn=collate_fn)

    model     = CVRegressor(model_path, dropout=BEST_PARAMS['dropout']).to(device)
    criterion = DualLoss(class_weights_tensor, aux_weight=BEST_PARAMS['aux_weight'])
    save_path = SAVE_DIR / f'{model_name.replace("-","_")}_best.pt'

    # Phase 1: Freeze BERT
    freeze_eps = BEST_PARAMS['freeze_epochs']
    print(f'\n  🧊 Phase 1: Freeze ({freeze_eps} epochs)')
    for param in model.bert.parameters(): param.requires_grad = False
    opt_frozen = torch.optim.AdamW(
        list(model.shared.parameters())+list(model.reg_head.parameters())+
        list(model.cls_head.parameters())+list(model.att_pool.parameters()),
        lr=BEST_PARAMS['head_lr'], weight_decay=BEST_PARAMS['weight_decay']
    )
    sched_frozen = get_linear_schedule_with_warmup(opt_frozen, 0, len(train_dl)*freeze_eps)
    history, best_mae = [], float('inf')

    for epoch in range(freeze_eps):
        tr = run_epoch(model, train_dl, opt_frozen, criterion, sched_frozen, train=True,
                       max_grad_norm=BEST_PARAMS['max_grad_norm'])
        vl = run_epoch(model, val_dl,   opt_frozen, criterion, train=False)
        history.append({**{f't_{k}':tr[k] for k in ['loss','mae','acc']},
                         **{f'v_{k}':vl[k] for k in ['loss','mae','acc']},
                         'lr':opt_frozen.param_groups[0]['lr'],'phase':'frozen'})
        print(f'  🧊 Ep{epoch+1:02d}  loss={vl["loss"]:.4f}  mae={vl["mae"]:.2f}  acc={vl["acc"]*100:.1f}%')
        if vl['mae'] < best_mae:
            best_mae = vl['mae']
            torch.save({'model_state':model.state_dict(),'model_name':model_name,
                        'model_path':model_path,'params':BEST_PARAMS,
                        'thresholds':(T1,T2,T3),'score_range':(SCORE_MIN,SCORE_MAX),
                        'best_epoch':epoch+1,'best_mae':best_mae}, save_path)

    # Phase 2: Unfreeze
    remaining_eps = FULL_EPOCHS - freeze_eps
    print(f'\n  🔥 Phase 2: Unfreeze all ({remaining_eps} epochs)')
    for param in model.bert.parameters(): param.requires_grad = True
    optimizer    = build_optimizer(model, BEST_PARAMS['lr'], BEST_PARAMS['head_lr'],
                                    BEST_PARAMS['weight_decay'], BEST_PARAMS['lr_decay'])
    total_steps  = len(train_dl)*remaining_eps
    warmup_steps = int(total_steps*BEST_PARAMS['warmup_ratio'])
    scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    patience_ctr = 0

    for epoch in range(remaining_eps):
        tr = run_epoch(model, train_dl, optimizer, criterion, scheduler, train=True,
                       max_grad_norm=BEST_PARAMS['max_grad_norm'])
        vl = run_epoch(model, val_dl,   optimizer, criterion, train=False)
        cur_lr = optimizer.param_groups[-1]['lr']
        history.append({**{f't_{k}':tr[k] for k in ['loss','mae','acc']},
                         **{f'v_{k}':vl[k] for k in ['loss','mae','acc']},
                         'lr':cur_lr,'phase':'unfrozen'})
        flag = '💾' if vl['mae'] < best_mae else '  '
        print(f'  {flag} 🔥 Ep{epoch+freeze_eps+1:02d}  '
              f'loss={vl["loss"]:.4f}  mae={vl["mae"]:.2f}  '
              f'acc={vl["acc"]*100:.1f}%  lr={cur_lr:.2e}')
        if vl['mae'] < best_mae:
            best_mae=vl['mae']; patience_ctr=0
            torch.save({'model_state':model.state_dict(),'model_name':model_name,
                        'model_path':model_path,'params':BEST_PARAMS,
                        'thresholds':(T1,T2,T3),'score_range':(SCORE_MIN,SCORE_MAX),
                        'best_epoch':epoch+freeze_eps+1,'best_mae':best_mae}, save_path)
        else:
            patience_ctr+=1
            if patience_ctr >= PATIENCE: print(f'  ⏹️  Early stopping'); break

    print(f'  ✅ Best MAE={best_mae:.2f}')
    final_models[model_name]=model; final_history[model_name]=history

print('\n✅ Training complete!')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('📈 Training Curves', fontsize=13, fontweight='bold')
for (val_k, train_k, label, ax) in [
    ('v_loss','t_loss','Loss',     axes[0]),
    ('v_mae', 't_mae', 'MAE',      axes[1]),
    ('v_acc', 't_acc', 'Accuracy', axes[2]),
]:
    for mn, hist in final_history.items():
        x = range(1, len(hist)+1)
        ax.plot(x, [h[val_k]   for h in hist], '-o', ms=4, label=f'{mn} val',   color=CLRS[mn], lw=2)
        ax.plot(x, [h[train_k] for h in hist], '--',       label=f'{mn} train', color=CLRS[mn], alpha=.4)
    ax.axvline(BEST_PARAMS['freeze_epochs']+.5, color='gray', linestyle=':', lw=1.5)
    ax.set_xlabel('Epoch'); ax.set_title(label); ax.legend(fontsize=8)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout(); plt.savefig('02_training.png', dpi=150, bbox_inches='tight'); plt.show()

---
## 7️⃣ Evaluation + Ensemble

In [ ]:
eval_results = {}

for model_name, model_path in MODELS.items():
    ckpt  = torch.load(SAVE_DIR/f'{model_name.replace("-","_")}_best.pt',
                       map_location=device, weights_only=False)
    model = CVRegressor(model_path, dropout=BEST_PARAMS['dropout']).to(device)
    model.load_state_dict(ckpt['model_state'])
    final_models[model_name] = model

    criterion = DualLoss(class_weights_tensor, aux_weight=BEST_PARAMS['aux_weight'])
    test_dl   = DataLoader(CVDatasetSliding(test_df, tokenizers[model_name]),
                           batch_size=8, collate_fn=collate_fn)
    res = run_epoch(model, test_dl, None, criterion, train=False)
    res['r2']       = r2_score(res['true_scores'], res['pred_scores'])
    res['spearman'] = spearmanr(res['true_scores'], res['pred_scores'])[0]
    eval_results[model_name] = res
    print(f'📊 {model_name:<12}  MAE={res["mae"]:.2f}  Acc={res["acc"]*100:.1f}%  '
          f'R²={res["r2"]:.3f}  (best_ep={ckpt["best_epoch"]})')
    print(classification_report(res['true_classes'], res['pred_classes'],
                                 target_names=CLASS_NAMES, labels=CLASS_LABELS))

# Ensemble
maes    = {mn: eval_results[mn]['mae'] for mn in MODELS}
inv_mae = {mn: 1/v for mn,v in maes.items()}
total   = sum(inv_mae.values())
weights = {mn: v/total for mn,v in inv_mae.items()}

ens_preds = np.zeros(len(test_df))
for mn in MODELS:
    ens_preds += weights[mn] * np.array(eval_results[mn]['pred_scores'])
true_scores = eval_results[list(MODELS.keys())[0]]['true_scores']

ens_cls = [score_to_class(p) for p in ens_preds]
true_cls= [score_to_class(s) for s in true_scores]
eval_results['Ensemble'] = {
    'mae':mean_absolute_error(true_scores,ens_preds),
    'rmse':np.sqrt(mean_squared_error(true_scores,ens_preds)),
    'r2':r2_score(true_scores,ens_preds),
    'spearman':spearmanr(true_scores,ens_preds)[0],
    'acc':sum(p==t for p,t in zip(ens_cls,true_cls))/len(true_cls),
    'pred_scores':ens_preds.tolist(),'true_scores':true_scores,
    'pred_classes':ens_cls,'true_classes':true_cls
}

winner = min(eval_results, key=lambda x: eval_results[x]['mae'])
print(f'\n🔀 Ensemble  MAE={eval_results["Ensemble"]["mae"]:.2f}  Acc={eval_results["Ensemble"]["acc"]*100:.1f}%')
print(f'🏆 Winner: {winner}')

In [ ]:
rows=[]
for mn,r in eval_results.items():
    rows.append({'Model':mn,'MAE':round(r['mae'],2),'RMSE':round(r['rmse'],2),
                 'R²':round(r['r2'],4),'Spearman':round(r['spearman'],4),
                 'Accuracy':f"{r['acc']*100:.1f}%",
                 '':('🏆' if mn==winner else ('🔀' if mn=='Ensemble' else ''))})
display(pd.DataFrame(rows).style.hide(axis='index')
        .highlight_min(subset=['MAE','RMSE'],color='#DCFCE7')
        .highlight_max(subset=['R²','Spearman'],color='#DCFCE7')
        .set_caption('📊 Results')
        .set_table_styles([{'selector':'caption','props':[('font-weight','bold'),('font-size','14px')]}]))

# Confusion matrices
all_names = list(eval_results.keys())
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('📊 Confusion Matrices', fontweight='bold')
for ax, mn in zip(axes, all_names):
    res = eval_results[mn]
    cm  = confusion_matrix(res['true_classes'],res['pred_classes'],labels=CLASS_LABELS)
    pct = cm.astype(float)/cm.sum(axis=1,keepdims=True)*100
    ann = np.array([[f'{v}\n({p:.0f}%)' for v,p in zip(rv,rp)] for rv,rp in zip(cm,pct)])
    sns.heatmap(cm, annot=ann, fmt='', ax=ax, cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                linewidths=.5, linecolor='white')
    title = ('🏆 ' if mn==winner else '')+f'{mn}\nMAE={res["mae"]:.2f}  Acc={res["acc"]*100:.1f}%'
    ax.set_title(title); ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout(); plt.savefig('03_confusion.png', dpi=150, bbox_inches='tight'); plt.show()

---
## 8️⃣ Explainability & Inference

In [ ]:
def get_word_importance(text, model, tokenizer, top_k=15):
    model.eval()
    enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
    ids = enc['input_ids'].to(device); msk = enc['attention_mask'].to(device)
    bert_eager = AutoModel.from_pretrained(
        model.bert.config._name_or_path, attn_implementation='eager').to(device)
    bert_eager.load_state_dict(model.bert.state_dict()); bert_eager.eval()
    with torch.no_grad():
        out = bert_eager(input_ids=ids, attention_mask=msk, output_attentions=True)
        if not out.attentions: del bert_eager; return [], []
        cls_att = out.attentions[-1][0][:,0,:].mean(0).cpu().numpy()
    del bert_eager
    tokens = tokenizer.convert_ids_to_tokens(ids[0].cpu().tolist())
    words,scores=[],[]; cw,cs='',0
    for tok,s in zip(tokens[1:-1],cls_att[1:-1]):
        if tok.startswith('##'): cw+=tok[2:]; cs=max(cs,s)
        else:
            if cw: words.append(cw); scores.append(cs)
            cw,cs=tok,s
    if cw: words.append(cw); scores.append(cs)
    pairs=sorted(zip(words,scores),key=lambda x:-x[1])[:top_k]
    return zip(*pairs) if pairs else ([],[])

SUGGESTION_TEMPLATES = {
    'ضعيف' :[(lambda cv,kws:len(kws)<3,'مهارات المجال ناقصة → أضف ≥5 مهارات تقنية'),
              (lambda cv,kws:'الخبرات' not in cv,'لا يوجد قسم خبرة'),
              (lambda cv,kws:'المهارات' not in cv,'قسم المهارات غائب'),
              (lambda cv,kws:len(cv.split())<150,'CV قصير → 300-500 كلمة'),
              (lambda cv,kws:True,'استخدم إنجازات ملموسة')],
    'متوسط':[(lambda cv,kws:len(kws)<5,'أضف مهارات مجال محددة لرفع الـ ATS'),
              (lambda cv,kws:'github' not in cv.lower() and 'linkedin' not in cv.lower(),'أضف رابط GitHub أو LinkedIn'),
              (lambda cv,kws:'شهادة' not in cv,'أضف شهادات احترافية'),
              (lambda cv,kws:True,'رتّب خبراتك عكسياً زمنياً'),
              (lambda cv,kws:True,'أضف أرقاماً لقياس الإنجازات')],
    'جيد'  :[(lambda cv,kws:len(kws)<8,'زد الكلمات المفتاحية لتحسين ATS'),
              (lambda cv,kws:'مشروع' not in cv,'أضف قسم مشاريع'),
              (lambda cv,kws:True,'خصّص الـ CV لكل وظيفة')],
    'ممتاز':[(lambda cv,kws:True,'CV ممتاز! راجع التنسيق للتوافق مع ATS'),
              (lambda cv,kws:True,'أضف قسم توصيات مهنية')]
}

def analyze_cv(arabic_cv, category, use_ensemble=True):
    cv_clean = clean_arabic_text(arabic_cv)
    ats_val, checks, matched_kws = calculate_ats_score(arabic_cv, category)
    bucket     = 'ats_low' if ats_val<50 else ('ats_mid' if ats_val<75 else 'ats_high')
    input_text = f'{category} {bucket} [SEP] {cv_clean}'

    def _predict(mn):
        tok = tokenizers[mn]; mdl = final_models[mn]
        ds  = CVDatasetSliding(pd.DataFrame([{'input_text':input_text,
                                              'suitability_score':SCORE_MIN,'category':category}]), tok)
        item = ds[0]
        mdl.eval()
        with torch.no_grad():
            pred_s, _ = mdl(item['chunks_ids'].unsqueeze(0).to(device),
                            item['chunks_mask'].unsqueeze(0).to(device),[item['n_chunks']])
        return unscale_score(float(pred_s.cpu().item()))

    if use_ensemble:
        pred_score = sum(weights[mn]*_predict(mn) for mn in MODELS)
        used_model = 'Ensemble'
    else:
        best_mn    = winner if winner!='Ensemble' else 'CAMeL-BERT'
        pred_score = _predict(best_mn)
        used_model = best_mn

    class_name  = CLASS_NAMES[score_to_class(pred_score)]
    missing_kws = [k for k in DOMAIN_KEYWORDS.get(category,DEFAULT_KEYWORDS) if k not in matched_kws][:5]
    suggestions = []
    for cond_fn,t in SUGGESTION_TEMPLATES.get(class_name,[]):
        if cond_fn(arabic_cv,matched_kws): suggestions.append(f'📌 {t}')
    for check,passed in checks.items():
        if not passed: suggestions.append(f'⚠️  {ATS_MESSAGES[check]}')

    best_mn = winner if winner!='Ensemble' else 'CAMeL-BERT'
    words,wscores = get_word_importance(input_text,final_models[best_mn],tokenizers[best_mn])
    return {'model':used_model,'predicted_score':round(pred_score,1),
            'predicted_class':class_name,'thresholds':f'{T1:.0f}/{T2:.0f}/{T3:.0f}',
            'ats_score':ats_val,'matched_keywords':matched_kws,'missing_keywords':missing_kws,
            'suggestions':suggestions[:7],
            'top_words':list(zip(words,[round(float(s),4) for s in wscores]))}

# Sample
sample = test_df.iloc[2]
result = analyze_cv(sample['arabic_cv'], sample['category'])
out_df = pd.DataFrame([
    ['Model',           result['model']],
    ['Category',        sample['category']],
    ['True Score',      f"{sample['suitability_score']:.1f} → {CLASS_NAMES[score_to_class(sample['suitability_score'])]}"],
    ['Predicted Score', f"{result['predicted_score']} → {result['predicted_class']}"],
    ['Thresholds',      result['thresholds']],
    ['ATS Score',       f"{result['ats_score']}/100"],
    ['Matched KWs',     ', '.join(result['matched_keywords']) or 'None'],
],columns=['Field','Value'])
display(out_df.style.hide(axis='index').set_caption('🔍 CV Analysis'))
print('\n💡 Suggestions:')
for i,s in enumerate(result['suggestions'],1): print(f'  {i}. {s}')

---
## 9️⃣ 💾 Save for API

In [ ]:
for mn,tok in tokenizers.items():
    tok.save_pretrained(SAVE_DIR/f'{mn.replace("-","_")}_tokenizer')
    print(f'✅ Tokenizer: {mn}')

config = {
    'thresholds'      : {'T1':T1,'T2':T2,'T3':T3},
    'score_range'     : {'min':SCORE_MIN,'max':SCORE_MAX},
    'class_names'     : CLASS_NAMES,'class_labels':CLASS_LABELS,
    'winner_model'    : winner,'models':MODELS,
    'hyperparams'     : {**BEST_PARAMS,'lr':str(BEST_PARAMS['lr']),'head_lr':str(BEST_PARAMS['head_lr'])},
    'ensemble_weights': {mn:round(w,4) for mn,w in weights.items()},
    'eval_summary'    : {mn:{'mae':round(r['mae'],4),'rmse':round(r['rmse'],4),
                              'r2':round(r['r2'],4),'spearman':round(r['spearman'],4),
                              'acc':round(r['acc'],4)} for mn,r in eval_results.items()}
}
with open(SAVE_DIR/'config.json','w',encoding='utf-8') as f:
    json.dump(config,f,ensure_ascii=False,indent=2)
print('✅ config.json')

with open(SAVE_DIR/'ats_config.json','w',encoding='utf-8') as f:
    json.dump({'domain_keywords':DOMAIN_KEYWORDS,'default_keywords':DEFAULT_KEYWORDS,
               'ats_messages':ATS_MESSAGES},f,ensure_ascii=False,indent=2)
print('✅ ats_config.json')

with open(SAVE_DIR/'eval_results.pkl','wb') as f:
    pickle.dump(eval_results,f)
print('✅ eval_results.pkl')

model_code = '''
"""Model classes — import in your API"""
import torch, torch.nn as nn, torch.nn.functional as F
from transformers import AutoModel

class AttentionPooling(nn.Module):
    def __init__(self, hidden_size=768):
        super().__init__()
        self.attention=nn.Sequential(nn.Linear(hidden_size,256),nn.Tanh(),nn.Linear(256,1))
    def forward(self, cls_embeddings, n_chunks):
        pooled=[]
        for i in range(cls_embeddings.shape[0]):
            n=n_chunks[i]; emb=cls_embeddings[i,:n,:]
            att=F.softmax(self.attention(emb),dim=0)
            pooled.append((att*emb).sum(dim=0))
        return torch.stack(pooled)

class CVRegressor(nn.Module):
    def __init__(self, model_path, dropout=0.2, n_classes=4):
        super().__init__()
        self.bert     = AutoModel.from_pretrained(model_path)
        self.att_pool = AttentionPooling(768)
        self.shared   = nn.Sequential(
            nn.Linear(768,256),nn.GELU(),nn.Dropout(dropout),
            nn.Linear(256,64), nn.GELU(),nn.Dropout(dropout))
        self.reg_head = nn.Sequential(nn.Linear(64,1),nn.Sigmoid())
        self.cls_head = nn.Linear(64, n_classes)
    def forward(self, chunks_ids, chunks_mask, n_chunks):
        B,C,L=chunks_ids.shape
        out=self.bert(input_ids=chunks_ids.view(B*C,L),
                      attention_mask=chunks_mask.view(B*C,L),output_attentions=False)
        cls=out.last_hidden_state[:,0,:].view(B,C,768)
        pooled=self.att_pool(cls,n_chunks)
        shared=self.shared(pooled)
        return self.shared(pooled), self.reg_head(shared).squeeze(-1), self.cls_head(shared)
'''
with open(SAVE_DIR/'model_classes.py','w',encoding='utf-8') as f:
    f.write(model_code)
print('✅ model_classes.py')

rows=[{'File':str(p.relative_to(SAVE_DIR)),
       'Size':f'{p.stat().st_size/1024/1024:.1f} MB' if p.stat().st_size>1e6 else f'{p.stat().st_size/1024:.0f} KB'}
      for p in sorted(SAVE_DIR.rglob('*')) if p.is_file()]
display(pd.DataFrame(rows).style.hide(axis='index').set_caption('📦 saved_model/'))
print(f'\n🎉 Done! Winner:{winner}  MAE={eval_results[winner]["mae"]:.2f}  Acc={eval_results[winner]["acc"]*100:.1f}%')